In [ ]:
# step 1: Import libraries and load model
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

I0000 00:00:1792418450.620211   15354 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792418450.900599   15354 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792418453.550199   15354 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
# Load the IMDB word index
word_index = imdb.get_word_index()
reverse_word_index = {value : key for key, value in word_index.items()}

In [4]:
# Load the pre-trained model with ReLU activation
from tensorflow.keras.models import load_model

model = load_model('Simple_rnn_imdb.h5')
model.summary()

I0000 00:00:1792418703.422810   15354 gpu_device.cc:2036] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1763 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2050, pci bus id: 0000:01:00.0, compute capability: 8.6


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 500, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,313,027 (5.01 MB)

 Trainable params: 1,313,025 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [5]:
model.get_weights()

[array([[ 0.07563742,  0.13604765,  0.9361162 , ...,  0.12785241,
         -0.1278421 , -0.11283714],
        [-0.00379839, -0.0436414 ,  0.04875347, ...,  0.02084359,
         -0.02899697, -0.04487544],
        [ 0.08532968,  0.00966924, -0.02012855, ..., -0.13794133,
          0.16254215,  0.11696681],
        ...,
        [ 0.20619361,  0.00457427,  0.20296587, ...,  0.20192541,
         -0.10423709, -0.14195582],
        [-0.084699  ,  0.03547462, -0.06829581, ..., -0.01274014,
         -0.02603446,  0.10204366],
        [-0.08356299, -0.03312313,  0.11854271, ...,  0.09198207,
         -0.17862569, -0.13053736]], shape=(10000, 128), dtype=float32),
 array([[-0.04289605, -0.02700347,  0.00751653, ..., -0.09250508,
         -0.04084595,  0.06801433],
        [-0.09195518, -0.00041486, -0.1023002 , ..., -0.14372407,
         -0.07597163, -0.10955924],
        [ 0.10320025, -0.06743977, -0.16686   , ...,  0.11072724,
         -0.11053308, -0.06147646],
        ...,
        [-0.0754049

In [6]:
# step 2: Helper Functions
# Function to decode reviews
def decode_review(encoded_review):
    return ' '.join([reverse_word_index.get(i - 3, '?') for i in encoded_review])

# Function to preprocess user input
def preprocess_text(text):
    words = text.lower().split()
    encoded_review = [word_index.get(word, 2) + 3 for word in words]
    padded_review = sequence.pad_sequences([encoded_review], maxlen=500)
    return padded_review

In [7]:
## Prediction function
def predict_sentiment(review):
    preprocessed_input = preprocess_text(review)
    prediction = model.predict(preprocessed_input)
    sentiment = 'Positive' if prediction[0][0] > 0.5 else 'Negative'
    return sentiment, prediction[0][0]

In [10]:
# step 4: User Input and Prediction
# Example review for prediction
example_review = "This movie was fantastic! The acting was great and the plot was thrilling."

sentiment, score = predict_sentiment(example_review)

print(f'Review: {example_review}')
print(f'Sentiment: {sentiment}')
print(f'Prediction: {score}')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step
Review: This movie was fantastic! The acting was great and the plot was thrilling.
Sentiment: Positive
Prediction: 0.6333378553390503
